# Notebook 04: Vectorization and Performance

**Module:** 01 Python Revision  
**Duration:** ~1.5 hours

---

## Learning Objectives

1. Understand why vectorization matters for ML
2. Benchmark Python loops vs NumPy operations
3. Implement vectorized distance computations (KNN preview)
4. Use random seeds for reproducibility
5. Connect performance to production inference scale

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. Intuition: Why Vectorization?

ML trains on millions of numbers. Python loops interpret each operation. NumPy pushes work to compiled C/Fortran.

At inference scale (e.g., sliding window over a 10000×10000 GeoTIFF in water-bodies-detection), vectorization is the difference between seconds and hours.

## 2. Loop vs Vectorized: Element-wise Sum

In [ ]:
n = 1_000_000
a = rng.random(n)
b = rng.random(n)

# Python loop
start = time.perf_counter()
result_loop = sum(a[i] + b[i] for i in range(n))
t_loop = time.perf_counter() - start

# NumPy vectorized
start = time.perf_counter()
result_vec = (a + b).sum()
t_vec = time.perf_counter() - start

print(f'Loop result:       {result_loop:.4f}')
print(f'Vectorized result: {result_vec:.4f}')
print(f'Loop time:         {t_loop:.4f}s')
print(f'Vectorized time:   {t_vec:.6f}s')
print(f'Speedup:           {t_loop/t_vec:.0f}x')

## 3. Loop vs Vectorized: Euclidean Distance

K-Nearest Neighbors (Module 03, your Day - 4) requires computing distances between points. Vectorization is essential.

In [ ]:
def euclidean_loop(a, b):
    """Distance between two vectors using Python loop."""
    return sum((a[i] - b[i]) ** 2 for i in range(len(a))) ** 0.5

def euclidean_vec(a, b):
    """Distance using NumPy."""
    return np.sqrt(np.sum((a - b) ** 2))

p = rng.random(1000)
q = rng.random(1000)

assert abs(euclidean_loop(p, q) - euclidean_vec(p, q)) < 1e-10
print(f'Distance (loop): {euclidean_loop(p, q):.6f}')
print(f'Distance (vec):  {euclidean_vec(p, q):.6f}')

## 4. Vectorized Distance Matrix (KNN Preview)

Given query point q and training matrix X (n_samples × n_features), compute distances to ALL training points at once.

$$d_i = \|\mathbf{x}_i - \mathbf{q}\|_2 = \sqrt{\sum_j (x_{ij} - q_j)^2}$$

Vectorized using broadcasting:

In [ ]:
def distance_matrix(X, q):
    """
    Compute Euclidean distance from query q to all rows of X.
    X: (n_samples, n_features)
    q: (n_features,)
    Returns: (n_samples,) distances
    """
    diff = X - q  # broadcasting: (n, d) - (d,) → (n, d)
    return np.sqrt((diff ** 2).sum(axis=1))

# Test
X_train = rng.normal(0, 1, (1000, 5))
q = rng.normal(0, 1, 5)

dists = distance_matrix(X_train, q)
print(f'Distances shape: {dists.shape}')
print(f'Min distance: {dists.min():.4f}')
print(f'Max distance: {dists.max():.4f}')

# Find 5 nearest neighbors
k = 5
nearest_idx = np.argsort(dists)[:k]
print(f'5 nearest indices: {nearest_idx}')
print(f'Their distances: {dists[nearest_idx]}')

## 5. Pairwise Distance Matrix

Compute distances between ALL pairs — used in clustering (Module 03).

In [ ]:
def pairwise_distances(X):
    """
    Compute pairwise Euclidean distances.
    X: (n, d) → returns (n, n) distance matrix
    
    Uses: ||a - b||² = ||a||² + ||b||² - 2(a·b)
    """
    sq_norms = (X ** 2).sum(axis=1)
    dists_sq = sq_norms[:, None] + sq_norms[None, :] - 2 * (X @ X.T)
    dists_sq = np.maximum(dists_sq, 0)  # numerical stability
    return np.sqrt(dists_sq)

X_small = rng.normal(0, 1, (50, 3))
D = pairwise_distances(X_small)
print(f'Pairwise distance matrix shape: {D.shape}')
print(f'Diagonal (should be 0): {D.diagonal()[:5]}')

## 6. Reproducibility with Random Seeds

ML experiments must be reproducible. Always set seeds:

In [ ]:
# NumPy random generator with seed
rng1 = np.random.default_rng(42)
print('Run 1:', rng1.random(3))

rng2 = np.random.default_rng(42)
print('Run 2:', rng2.random(3))  # identical

# Different seed → different results
rng3 = np.random.default_rng(0)
print('Run 3:', rng3.random(3))

## 7. In-place Operations and Memory

For large arrays, avoid unnecessary copies:

In [ ]:
a = np.ones(1_000_000)
b = a  # b is a VIEW, not a copy
b += 1  # modifies a too!
print(f'a[0] after b+=1: {a[0]}')  # 2.0

a = np.ones(1_000_000)
b = a.copy()  # explicit copy
b += 1
print(f'a[0] after copy b+=1: {a[0]}')  # 1.0

## 8. Benchmark Visualization

In [ ]:
sizes = [100, 1000, 10000, 100000, 500000]
loop_times = []
vec_times = []

for n in sizes:
    a = rng.random(n)
    b = rng.random(n)
    
    start = time.perf_counter()
    _ = sum(a[i] + b[i] for i in range(n))
    loop_times.append(time.perf_counter() - start)
    
    start = time.perf_counter()
    _ = (a + b).sum()
    vec_times.append(time.perf_counter() - start)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, loop_times, 'o-', label='Python loop', color='red')
ax.plot(sizes, vec_times, 's-', label='NumPy vectorized', color='blue')
ax.set_xlabel('Array size')
ax.set_ylabel('Time (seconds)')
ax.set_title('Loop vs Vectorized Performance')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Production Consideration

In **water-bodies-detection** `predict.py`, sliding-window inference processes thousands of 512×512 tiles. Each tile operation uses vectorized NumPy/PyTorch ops. A Python loop over pixels would make inference unusable.

Rule: **If you're writing a for-loop over data points in ML, you're probably doing it wrong.**

## Exercise 6: Vectorized Softmax

Implement softmax vectorized (preview of Module 05):

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Input: 1D array of logits. Output: probabilities summing to 1.

In [ ]:
# YOUR CODE HERE
def softmax(z):
    pass

z = np.array([2.0, 1.0, 0.1])
probs = softmax(z)
print('Softmax:', probs)
print('Sum:', probs.sum())  # should be 1.0

## Exercise 7: Vectorized Accuracy

Given arrays `y_true` and `y_pred`, compute accuracy without loops:

$$\text{accuracy} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}[y_{\text{true},i} = y_{\text{pred},i}]$$

In [ ]:
# YOUR CODE HERE
y_true = np.array([0, 1, 1, 0, 1, 0, 1, 1, 0, 0])
y_pred = np.array([0, 1, 0, 0, 1, 1, 1, 1, 0, 0])

# accuracy = ?


## Interview Questions

1. Why is vectorization faster than Python loops?
2. How would you compute distances from one point to 1 million training points efficiently?
3. Why set random seeds in ML experiments?
4. What is the difference between a view and a copy in NumPy?

---

## Module 01 Complete

You now have the Python/numerical foundation for all subsequent modules.

**Before Module 02:** Complete exercises, assignment, quiz, and gate questions.

See: [CHEATSHEET.md](CHEATSHEET.md) | [quiz/module_01_quiz.md](quiz/module_01_quiz.md) | [exercises/README.md](exercises/README.md)